# 07.01 | Joint categorical HRV profiles

**Study:** Heart rate variability in older adults from southern Chile  
**Phase:** 07 — Two-factor HRV profiling  
**Notebook:** 01 — Complete descriptive atlas of existing categorical intersections

## Research objective

Characterize the joint distributions of **five primary HRV measures** (SDNN, RMSSD, HF, LF, VLF) and **two complementary cardiac measures** (Mean RR, Mean HR) across **all ten unique pairs** of five accepted categorical factors:

- Age group (`age_group`): `60-69`, `70-79`, `80+`.
- Sex (`sex_category`): `Female`, `Male`.
- Nutritional status (`bmi_category`): `Underweight`, `Normal`, `Overweight`, `Obesity` (previously accepted older-adult MINSAL categories).
- Systolic blood pressure category (`sbp_category`): `<140 mmHg`, `>=140 mmHg`.
- Diastolic blood pressure category (`dbp_category`): `<90 mmHg`, `>=90 mmHg`.

**Scope:** exhaustive *descriptive* profiles, not interaction inference or a search for significance. Every accepted source row stays unchanged. Empty category intersections are explicitly represented with `n=0` and undefined statistics, rather than fabricated values. Small occupied groups remain visible and are labeled, without arbitrary exclusion or category merging.

**Expected deliverables when run:** full joint cell-count table, support inventory, 7-outcome quantitative profiles for every possible cell (including empty ones), cohort reference values, publication-oriented figures, a figure manifest, run metadata, a comprehensive results report, and a methodological README. Only this notebook is distributed as the downloadable artifact; all outputs are generated when it runs.

**Relationship to 06:** stages 06.01–06.04 describe one-factor associations; 07.01 examines observed two-factor distributions only. Conditional comparisons and conditional continuous associations belong to 07.02 and 07.03, respectively.

## 0. Configuration

Run from the project directory or any of its descendants. Set `INPUT_FILE_OVERRIDE` only when `outputs/dataset_final.csv` cannot be resolved automatically. Outputs stay within the dedicated stage-07 folder; no upstream file is edited. Displaying all figures inline is disabled by default because the complete atlas contains many high-resolution images.

In [1]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

INPUT_FILE_OVERRIDE = None    # e.g. "/absolute/path/to/outputs/dataset_final.csv"
OUTPUT_ROOT_OVERRIDE = None   # default: <project>/eda_results/hrv_associations/07_01_joint_categorical_hrv_profiles
SHOW_FIGURES = False         # True = also display all saved multipanel figures inline
PLOT_DPI = 210
SEED = 20261128              # reproducible jitter for very small subgroups; no statistical resampling

PRIMARY = {
    "t2m_pre_sdnn": "SDNN", "t2m_pre_rmssd": "RMSSD", "t2m_pre_hf": "HF",
    "t2m_pre_lf": "LF", "t2m_pre_vlf": "VLF",
}
COMPLEMENTARY = {"t2m_pre_mean_rr": "Mean RR", "t2m_pre_mean_hr": "Mean HR"}
MEASURES = {**PRIMARY, **COMPLEMENTARY}
VISUAL_LOG1P = {"t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf"}  # figures only

FACTOR_LEVELS = {
    "age_group": ["60-69", "70-79", "80+"],
    "sex_category": ["Female", "Male"],
    "bmi_category": ["Underweight", "Normal", "Overweight", "Obesity"],
    "sbp_category": ["<140 mmHg", ">=140 mmHg"],
    "dbp_category": ["<90 mmHg", ">=90 mmHg"],
}
FACTOR_LABELS = {
    "age_group": "Age group", "sex_category": "Sex",
    "bmi_category": "BMI category", "sbp_category": "SBP category",
    "dbp_category": "DBP category",
}
# Complete, nonduplicated factorial design, mirroring the categorical-pair inventory of stage 05.
PAIR_ORDER = [
    ("age_group", "sex_category"),
    ("age_group", "bmi_category"),
    ("sex_category", "bmi_category"),
    ("sbp_category", "dbp_category"),
    ("bmi_category", "sbp_category"),
    ("bmi_category", "dbp_category"),
    ("age_group", "sbp_category"),
    ("age_group", "dbp_category"),
    ("sex_category", "sbp_category"),
    ("sex_category", "dbp_category"),
]

NAVY, MUTED, TEAL, CORAL, BLUE = "#19394C", "#627887", "#247E90", "#C76D52", "#4E87A1"
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.01)
plt.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#DAE4EA", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": "#E7EDF1",
    "grid.alpha": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": False, "axes.titleweight": "semibold",
})
print("Python", platform.python_version(), "| pandas", pd.__version__, "| seaborn", sns.__version__)

Python 3.14.10 | pandas 3.1.1 | seaborn 0.13.2


## 1. Load the accepted analytical dataset

The accepted source CSV and all five existing categorical columns are read as-is. **No recoding, missingness-driven filtering, outcome replacement, category fusion, new cutoffs, or fixed-hash/count requirements** are introduced. A minimal required-column check merely reports missing variables if the incorrect CSV is supplied.

In [2]:
def find_project_input():
    if INPUT_FILE_OVERRIDE is not None:
        return Path(INPUT_FILE_OVERRIDE).expanduser().resolve()
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / "outputs" / "dataset_final.csv"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("outputs/dataset_final.csv not found; set INPUT_FILE_OVERRIDE.")

input_file = find_project_input()
project_root = input_file.parent.parent if input_file.parent.name == "outputs" else input_file.parent
output_root = (Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE
               else project_root / "eda_results" / "hrv_associations" / "07_01_joint_categorical_hrv_profiles")
tables_dir = output_root / "tables"
fig_dir = output_root / "figures" / "multipanels"
tables_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(input_file)
required = [*FACTOR_LEVELS, *MEASURES]
missing = sorted(set(required) - set(data.columns))
if missing:
    raise KeyError(f"Missing required columns in accepted CSV: {missing}")

print("Source:", input_file)
print("Source records:", len(data))
print("Categories:")
for factor, levels in FACTOR_LEVELS.items():
    print(" ", FACTOR_LABELS[factor], data[factor].value_counts().reindex(levels, fill_value=0).to_dict())
print("Output root:", output_root)

Source: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/outputs/dataset_final.csv
Source records: 402
Categories:
  Age group {'60-69': 184, '70-79': 190, '80+': 28}
  Sex {'Female': 294, 'Male': 108}
  BMI category {'Underweight': 17, 'Normal': 118, 'Overweight': 117, 'Obesity': 150}
  SBP category {'<140 mmHg': 298, '>=140 mmHg': 104}
  DBP category {'<90 mmHg': 354, '>=90 mmHg': 48}
Output root: /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_01_joint_categorical_hrv_profiles


## 2. Exhaustive cell inventory and support

All ten categorical pairs are evaluated across their **full Cartesian products**, including absent intersections. The two-factor inventory records joint `n`, cohort percentage, within-row and within-column percentage, marginal category counts, missing assignments (if present), and descriptive support flags.

The flags `n < 5` and `n < 10` are **interpretive warnings only**: no observation is excluded and no statistical significance or formal estimability cutoff is inferred. A cell of `n=0` has no measurable distribution; `n=1` has an observed value, but no estimable within-cell standard deviation.

In [3]:
def save_table(frame, name):
    path = tables_dir / name
    frame.to_csv(path, index=False, float_format="%.12g")
    return path

cell_rows = []
summary_rows = []
for factor_a, factor_b in PAIR_ORDER:
    a_levels, b_levels = FACTOR_LEVELS[factor_a], FACTOR_LEVELS[factor_b]
    joint = pd.crosstab(data[factor_a], data[factor_b]).reindex(
        index=a_levels, columns=b_levels, fill_value=0
    )
    n_complete = int(joint.to_numpy().sum())
    n_missing = int(len(data) - n_complete)
    occupied = joint.to_numpy()[joint.to_numpy() > 0]
    for cat_a, cat_b in [(a, b) for a in a_levels for b in b_levels]:
        n = int(joint.loc[cat_a, cat_b])
        row_n = int(joint.loc[cat_a, :].sum())
        col_n = int(joint.loc[:, cat_b].sum())
        cell_rows.append({
            "factor_a": factor_a, "factor_b": factor_b,
            "category_a": cat_a, "category_b": cat_b,
            "n": n, "joint_pct": 100 * n / n_complete if n_complete else np.nan,
            "row_pct": 100 * n / row_n if row_n else np.nan,
            "column_pct": 100 * n / col_n if col_n else np.nan,
            "row_n": row_n, "column_n": col_n,
            "n_pair_complete": n_complete, "n_pair_missing": n_missing,
            "is_empty": n == 0, "is_occupied_below_5": 0 < n < 5,
            "is_occupied_below_10": 0 < n < 10,
        })
    summary_rows.append({
        "factor_a": factor_a, "factor_b": factor_b,
        "comparison": f"{FACTOR_LABELS[factor_a]} × {FACTOR_LABELS[factor_b]}",
        "n_pair_complete": n_complete, "n_pair_missing": n_missing,
        "n_possible_cells": len(a_levels) * len(b_levels),
        "n_occupied_cells": int((joint > 0).to_numpy().sum()),
        "n_empty_cells": int((joint == 0).to_numpy().sum()),
        "n_occupied_cells_below_5": int(((joint > 0) & (joint < 5)).to_numpy().sum()),
        "n_occupied_cells_below_10": int(((joint > 0) & (joint < 10)).to_numpy().sum()),
        "min_occupied_n": int(occupied.min()) if len(occupied) else np.nan,
        "median_occupied_n": float(np.median(occupied)) if len(occupied) else np.nan,
        "max_occupied_n": int(occupied.max()) if len(occupied) else np.nan,
    })

cell_counts = pd.DataFrame(cell_rows)
support_summary = pd.DataFrame(summary_rows)
save_table(cell_counts, "categorical_pairwise_cell_counts.csv")
save_table(support_summary, "categorical_pairwise_support_summary.csv")
display(support_summary)
print("Potential cells:", int(support_summary.n_possible_cells.sum()),
      "| Occupied:", int(support_summary.n_occupied_cells.sum()),
      "| Empty:", int(support_summary.n_empty_cells.sum()))

,factor_a,factor_b,comparison,n_pair_complete,n_pair_missing,n_possible_cells,n_occupied_cells,n_empty_cells,n_occupied_cells_below_5,n_occupied_cells_below_10,min_occupied_n,median_occupied_n,max_occupied_n
0,age_group,sex_category,Age group × Sex,402,0,6,6,0,0,1,6,51.0,141
1,age_group,bmi_category,Age group × BMI category,402,0,12,12,0,2,3,1,29.5,80
2,sex_category,bmi_category,Sex × BMI category,402,0,8,8,0,1,1,2,40.5,112
3,sbp_category,dbp_category,SBP category × DBP category,402,0,4,4,0,0,0,15,52.0,283
4,bmi_category,sbp_category,BMI category × SBP category,402,0,8,8,0,1,1,3,36.5,105
5,bmi_category,dbp_category,BMI category × DBP category,402,0,8,7,1,0,1,9,24.0,126
6,age_group,sbp_category,Age group × SBP category,402,0,6,6,0,0,1,7,48.5,143
7,age_group,dbp_category,Age group × DBP category,402,0,6,5,1,0,0,17,31.0,173
8,sex_category,sbp_category,Sex × SBP category,402,0,4,4,0,0,0,31,75.0,221
9,sex_category,dbp_category,Sex × DBP category,402,0,4,4,0,0,0,18,60.0,264


Potential cells: 66 | Occupied: 64 | Empty: 2


## 3. Complete seven-measure distributions for every joint cell

For each cell and outcome, the following are reported **in original source units**: `n`, mean, sample SD (`ddof=1`), minimum, P05, P25, median, P75, P95, maximum and IQR. Empty cells receive `n=0` and undefined numerical summaries. The full Cartesian grid is retained even where the accepted cohort has no observations.

Each outcome appears in 66 cells in the accepted factor design (7 × 66 = 462 profile rows when all categories are present as defined). Of these, the occupied-cell profiles have actual observations. These are descriptive distributions; intervals, p-values, adjusted q-values, and factorial interaction estimators are intentionally outside the scope of 07.01.

In [4]:
def summarize_values(values):
    x = np.asarray(values, dtype=float)
    if x.size == 0:
        return {k: np.nan for k in ("mean", "sd", "minimum", "q05", "q25",
                                    "median", "q75", "q95", "maximum", "iqr")}
    q05, q25, q50, q75, q95 = np.quantile(x, [0.05, 0.25, 0.50, 0.75, 0.95])
    return {
        "mean": float(np.mean(x)), "sd": float(np.std(x, ddof=1)) if x.size >= 2 else np.nan,
        "minimum": float(np.min(x)), "q05": float(q05), "q25": float(q25),
        "median": float(q50), "q75": float(q75), "q95": float(q95),
        "maximum": float(np.max(x)), "iqr": float(q75 - q25),
    }

reference_rows = []
for outcome, label in MEASURES.items():
    reference_rows.append({"outcome": outcome, "outcome_label": label,
                           "outcome_class": "Primary HRV" if outcome in PRIMARY else "Complementary cardiac",
                           "n": len(data), **summarize_values(data[outcome].to_numpy(dtype=float))})
cohort_reference = pd.DataFrame(reference_rows)

profile_rows = []
for item in cell_counts.itertuples(index=False):
    factor_a, factor_b = item.factor_a, item.factor_b
    selected = data[(data[factor_a] == item.category_a) & (data[factor_b] == item.category_b)]
    n = len(selected)
    if n == 0:
        status = "empty"
    elif n == 1:
        status = "single_observation"
    elif n < 5:
        status = "sparse_2_to_4"
    elif n < 10:
        status = "small_5_to_9"
    else:
        status = "descriptive_n_ge_10"
    for outcome, label in MEASURES.items():
        profile_rows.append({
            "factor_a": factor_a, "factor_b": factor_b,
            "category_a": item.category_a, "category_b": item.category_b,
            "profile_id": f"{item.category_a} | {item.category_b}",
            "outcome": outcome, "outcome_label": label,
            "outcome_class": "Primary HRV" if outcome in PRIMARY else "Complementary cardiac",
            "n": n, "support_status": status,
            **summarize_values(selected[outcome].to_numpy(dtype=float)),
        })
profiles = pd.DataFrame(profile_rows)
save_table(cohort_reference, "cohort_hrv_reference.csv")
save_table(profiles, "joint_hrv_descriptive_profiles.csv")
print("Profile rows:", len(profiles),
      "| Nonempty profiles:", int((profiles.n > 0).sum()),
      "| Empty-cell rows:", int((profiles.n == 0).sum()))
display(profiles.head(12))

Profile rows: 462 | Nonempty profiles: 448 | Empty-cell rows: 14


,factor_a,factor_b,category_a,category_b,profile_id,outcome,outcome_label,outcome_class,n,support_status,mean,sd,minimum,q05,q25,median,q75,q95,maximum,iqr
0,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_sdnn,SDNN,Primary HRV,141,descriptive_n_ge_10,166.402128,68.673669,9.3,47.00,124.00,167.0,209.00,271.00,430.0,85.00
1,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_rmssd,RMSSD,Primary HRV,141,descriptive_n_ge_10,14.902837,6.945666,1.0,5.10,10.80,14.3,18.40,28.40,38.2,7.60
2,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_hf,HF,Primary HRV,141,descriptive_n_ge_10,489.056738,1329.971991,3.0,17.00,58.00,136.0,341.00,1640.00,12130.0,283.00
3,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_lf,LF,Primary HRV,141,descriptive_n_ge_10,560.539007,1336.515152,11.0,25.00,78.00,158.0,324.00,2040.00,9860.0,246.00
4,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_vlf,VLF,Primary HRV,141,descriptive_n_ge_10,110.921986,199.657638,2.0,6.00,20.00,43.0,109.00,530.00,1430.0,89.00
5,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_mean_rr,Mean RR,Complementary cardiac,141,descriptive_n_ge_10,842.276596,113.880709,589.0,670.00,768.00,832.0,929.00,1019.00,1296.0,161.00
6,age_group,sex_category,60-69,Female,60-69 | Female,t2m_pre_mean_hr,Mean HR,Complementary cardiac,141,descriptive_n_ge_10,72.574468,9.893168,46.0,59.00,65.00,72.0,78.00,90.00,102.0,13.00
7,age_group,sex_category,60-69,Male,60-69 | Male,t2m_pre_sdnn,SDNN,Primary HRV,43,descriptive_n_ge_10,84.439535,90.340678,4.8,7.07,14.85,59.0,144.00,218.00,355.0,129.15
8,age_group,sex_category,60-69,Male,60-69 | Male,t2m_pre_rmssd,RMSSD,Primary HRV,43,descriptive_n_ge_10,12.762791,6.831998,3.3,4.16,6.95,11.8,16.75,24.94,33.3,9.80
9,age_group,sex_category,60-69,Male,60-69 | Male,t2m_pre_hf,HF,Primary HRV,43,descriptive_n_ge_10,329.790698,641.078725,3.0,5.40,25.00,68.0,318.50,2269.50,2490.0,293.50


## 4. Figure system

Figures have three complementary roles:

1. **Cell-support heatmaps** show the actual sample size (`n`) and within-row percentage for every combination, explicitly marking `n=0`.
2. **Median atlases** show all seven observed original-unit medians using a *separate color scale for each outcome*. Colors must **not** be compared across panels; exact cell medians and cell sizes are annotated.
3. **Distribution atlases** present every observed joint subgroup for five primary HRV outcomes and separately for two complementary cardiac outcomes. Source data are preserved, with small groups explicitly flagged and their individual points displayed. Spectral HF/LF/VLF may be visualized as `log1p` where defined, **for figures only**. All exported statistics use original values.

The plot helpers reserve physical space for the main title, subtitle and subplot headings to avoid the text overlaps encountered in earlier stages. No arbitrary range restriction is applied to raw values.

In [5]:
figure_manifest = []

def figure_header(fig, title, subtitle=""):
    h = fig.get_figheight()
    fig.suptitle(title, x=0.058, y=1 - 0.16/h, ha="left", va="top",
                 fontsize=18, color=NAVY, weight="bold")
    if subtitle:
        fig.text(0.058, 1 - 0.64/h, subtitle, ha="left", va="top",
                 fontsize=9.6, color=MUTED)


def plot_area(fig, left=0.08, right=0.96, bottom_inches=0.82,
              top_inches=1.40, wspace=0.40, hspace=0.53):
    h = fig.get_figheight()
    fig.subplots_adjust(left=left, right=right,
                        bottom=bottom_inches/h, top=1-top_inches/h,
                        wspace=wspace, hspace=hspace)


def save_figure(fig, filename, comparison, role, description):
    target = fig_dir / filename
    fig.savefig(target, dpi=PLOT_DPI, bbox_inches="tight", pad_inches=0.17)
    figure_manifest.append({
        "figure": str(target.relative_to(output_root)),
        "comparison": comparison, "role": role, "description": description,
    })
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)


def pair_slug(factor_a, factor_b):
    aliases = {"age_group": "age", "sex_category": "sex", "bmi_category": "bmi",
               "sbp_category": "sbp", "dbp_category": "dbp"}
    return f"{aliases[factor_a]}_{aliases[factor_b]}"


def pair_title(factor_a, factor_b):
    return f"{FACTOR_LABELS[factor_a]} × {FACTOR_LABELS[factor_b]}"


def display_label(outcome):
    if outcome == "t2m_pre_mean_rr":
        return "Mean RR (source units)"
    if outcome == "t2m_pre_mean_hr":
        return "Mean HR (source units)"
    return MEASURES[outcome] + " (source units)"


def median_fmt(value):
    if not np.isfinite(value):
        return "—"
    return f"{value:,.0f}" if abs(value) >= 100 else f"{value:.1f}"


def heatmap_tick_style(ax):
    ax.tick_params(axis="both", length=0, labelsize=9)
    plt.setp(ax.get_xticklabels(), rotation=18, ha="right", rotation_mode="anchor")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.set_xlabel("")
    ax.set_ylabel("")

## 5. Exhaustive categorical cell-support visualizations

Ten heatmaps (one per pair) show the observed joint sample sizes, within-row percentages, and empty intersections. A separate support-overview figure summarizes the number of occupied, empty and small cells across all pairs.

In [6]:
def plot_support_overview():
    view = support_summary.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(view))
    fig, axes = plt.subplots(1, 2, figsize=(18.0, 7.2))
    ax = axes[0]
    ax.barh(y, view.n_occupied_cells, color=TEAL, label="Occupied")
    ax.barh(y, view.n_empty_cells, left=view.n_occupied_cells, color="#D4DFE4", label="Empty")
    ax.set_yticks(y, view.comparison, fontsize=9.2)
    ax.set_xlabel("Number of possible joint cells")
    ax.set_title("Available intersections", pad=15)
    ax.legend(loc="lower right", frameon=False, fontsize=9)
    ax.grid(axis="x", color="#E6EEF2")
    axes[1].barh(y, view.n_occupied_cells_below_10, color="#D2A05E", label="0 < n < 10")
    axes[1].barh(y, view.n_occupied_cells_below_5, color=CORAL, label="0 < n < 5")
    axes[1].set_yticks(y, [""] * len(y))
    axes[1].set_xlabel("Small occupied cells (descriptive warning only)")
    axes[1].set_title("Representation constraints", pad=15)
    axes[1].legend(loc="lower right", frameon=False, fontsize=9)
    axes[1].grid(axis="x", color="#E6EEF2")
    for a in axes:
        a.set_axisbelow(True)
        a.spines[["top", "right"]].set_visible(False)
    figure_header(fig, "Joint-profile sample support",
                  "All ten categorical pairs | N is unchanged | Flags identify precision constraints, not exclusions")
    plot_area(fig, left=0.25, right=0.975, top_inches=1.40, bottom_inches=0.75, wspace=0.25)
    save_figure(fig, "00_joint_profile_support_overview.png", "all_pairs", "support_overview",
                "Number of occupied, empty and sparsely represented cells for all ten categorical pairs")


def plot_count_heatmap(factor_a, factor_b, idx):
    part = cell_counts[(cell_counts.factor_a == factor_a) & (cell_counts.factor_b == factor_b)]
    levels_a, levels_b = FACTOR_LEVELS[factor_a], FACTOR_LEVELS[factor_b]
    counts = part.pivot(index="category_a", columns="category_b", values="n").reindex(
        index=levels_a, columns=levels_b).astype(int)
    fractions = part.pivot(index="category_a", columns="category_b", values="row_pct").reindex(
        index=levels_a, columns=levels_b)
    annotation = np.asarray([[f"{counts.iloc[i,j]}\n({fractions.iloc[i,j]:.1f}% of row)"
                             for j in range(len(levels_b))] for i in range(len(levels_a))], dtype=object)
    fig, ax = plt.subplots(figsize=(11.8, 2.9 + 1.2 * len(levels_a)))
    sns.heatmap(counts, annot=annotation, fmt="", cmap="Blues", linewidths=1.6,
                linecolor="white", vmin=0, cbar_kws={"label": "Observed records", "shrink": .80},
                ax=ax, annot_kws={"fontsize": 11})
    ax.set_xlabel(FACTOR_LABELS[factor_b], labelpad=13, fontsize=11)
    ax.set_ylabel(FACTOR_LABELS[factor_a], labelpad=12, fontsize=11)
    ax.tick_params(length=0, labelsize=10)
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
    plt.setp(ax.get_yticklabels(), rotation=0)
    figure_header(fig, pair_title(factor_a, factor_b) + " | Joint-cell sizes",
                  "Absolute n; percent within each first-factor row | Empty cells retained as n=0")
    plot_area(fig, left=0.17, right=0.90, bottom_inches=0.82, top_inches=1.45)
    save_figure(fig, f"{idx:02d}_{pair_slug(factor_a,factor_b)}_cell_counts.png",
                pair_title(factor_a, factor_b), "cell_counts",
                "Joint category counts and within-first-factor percentages, including empty cells")

plot_support_overview()
for idx, pair in enumerate(PAIR_ORDER, start=1):
    plot_count_heatmap(*pair, idx)
print("Cell-support figures:", len(figure_manifest))

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Cell-support figures: 11


## 6. Median atlas for all seven outcomes

Each outcome uses its own original-scale color range, so colors **cannot be compared across different HRV measures**. Each tile reports a median and its observed `n`; missing tiles are marked with an em dash. The reported medians are from the accepted source values, with no numerical transformation.

In [7]:
def plot_median_atlas(factor_a, factor_b, idx):
    a_levels, b_levels = FACTOR_LEVELS[factor_a], FACTOR_LEVELS[factor_b]
    part = profiles[(profiles.factor_a == factor_a) & (profiles.factor_b == factor_b)]
    fig, axes = plt.subplots(2, 4, figsize=(20.0, 11.9))
    for ax, (outcome, outcome_label) in zip(axes.flat, MEASURES.items()):
        f = part.loc[part.outcome == outcome]
        medians = f.pivot(index="category_a", columns="category_b", values="median").reindex(
            index=a_levels, columns=b_levels)
        counts = f.pivot(index="category_a", columns="category_b", values="n").reindex(
            index=a_levels, columns=b_levels)
        labels = np.asarray([[f"{median_fmt(medians.iloc[i,j])}\nn={int(counts.iloc[i,j])}"
                              if counts.iloc[i,j] > 0 else "—\nn=0"
                              for j in range(len(b_levels))] for i in range(len(a_levels))], dtype=object)
        ax.set_facecolor("#EDF2F4")
        val = medians.to_numpy(dtype=float)
        finite = val[np.isfinite(val)]
        if len(finite):
            lo, hi = float(finite.min()), float(finite.max())
            pad = max(abs(hi - lo) * 0.06, 0.0001)
            if lo == hi:
                pad = max(abs(lo) * 0.04, 0.5)
            sns.heatmap(medians, mask=medians.isna(), annot=labels, fmt="", ax=ax,
                        cmap="YlGnBu", vmin=lo-pad, vmax=hi+pad, linewidths=1.1,
                        linecolor="white", cbar=True,
                        cbar_kws={"shrink": .61, "pad": .02},
                        annot_kws={"fontsize": 8.8})
        else:
            ax.set_xlim(0, len(b_levels))
            ax.set_ylim(len(a_levels), 0)
        # Seaborn masks do not annotate missing cells: mark structural empties explicitly.
        for row_idx in range(len(a_levels)):
            for col_idx in range(len(b_levels)):
                if counts.iloc[row_idx, col_idx] == 0:
                    ax.text(col_idx + 0.5, row_idx + 0.5, "—\nn=0",
                            ha="center", va="center", fontsize=9, color=MUTED)
        ax.set_title(outcome_label + " | median", fontsize=12, pad=14)
        heatmap_tick_style(ax)
    axes.flat[-1].set_visible(False)
    figure_header(fig, pair_title(factor_a, factor_b) + " | Median atlas",
                  "Seven outcomes in source units | Each panel has its own color scale | Each tile shows median and n")
    plot_area(fig, left=0.062, right=0.965, bottom_inches=0.72, top_inches=1.37,
              wspace=0.61, hspace=0.28)
    save_figure(fig, f"{idx:02d}_{pair_slug(factor_a,factor_b)}_median_atlas.png",
                pair_title(factor_a, factor_b), "median_atlas",
                "Original-unit medians of five primary HRV and two complementary cardiac outcomes; cell n shown")

for idx, pair in enumerate(PAIR_ORDER, start=1):
    plot_median_atlas(*pair, idx)
print("Figures after median atlases:", len(figure_manifest))

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Figures after median atlases: 21


## 7. All-outcome distribution atlases

A complete point-and-box display is produced for every observed joint subgroup. The subgroups are always shown in the previously accepted category order. Empty groups appear in the cell tables and count figures; no distribution is invented for them.

**Small groups:** each occupied group shows its sample size on the y-axis; when `n < 5`, raw observations are highlighted with coral points and an asterisk in the group label. Boxplots are descriptive, particularly unreliable for cells containing one or two observations. The underlying data are never removed or trimmed. HF/LF/VLF may use `log1p` for **visualization only**, with explicit axis labels.

In [8]:
def plot_distribution_atlas(factor_a, factor_b, idx, subset):
    outcomes = PRIMARY if subset == "primary" else COMPLEMENTARY
    levels_a, levels_b = FACTOR_LEVELS[factor_a], FACTOR_LEVELS[factor_b]
    relevant = cell_counts[(cell_counts.factor_a == factor_a) & (cell_counts.factor_b == factor_b)]
    occupied = relevant[relevant.n > 0].copy()
    occupied["short_label"] = occupied.apply(
        lambda r: f"{r.category_a} · {r.category_b}  (n={int(r.n)})" + (" *" if r.n < 5 else ""), axis=1
    )
    occupied = occupied.reset_index(drop=True)
    ordered_keys = list(zip(occupied.category_a, occupied.category_b))
    labels = occupied.short_label.tolist()
    label_map = dict(zip(ordered_keys, labels))
    grouped_data = data.copy()
    grouped_data["_joint_label"] = [label_map.get((a,b), None)
                                      for a,b in zip(grouped_data[factor_a], grouped_data[factor_b])]

    n_outcomes = len(outcomes)
    n_rows = 3 if subset == "primary" else 1
    n_cols = 2
    height_per_panel = max(5.3, 0.42 * len(labels) + 1.55)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(19.8, 1.65 + height_per_panel * n_rows))
    axes_flat = np.asarray(axes).reshape(-1)
    jitter_rng = np.random.default_rng(SEED + idx)
    for ax, (outcome, outcome_name) in zip(axes_flat, outcomes.items()):
        plot_data = grouped_data[["_joint_label", outcome]].copy()
        values = plot_data[outcome].to_numpy(dtype=float)
        is_log = outcome in VISUAL_LOG1P and np.all(values > -1)
        plot_data["_visual_value"] = np.log1p(values) if is_log else values
        sns.boxplot(data=plot_data, y="_joint_label", x="_visual_value", order=labels,
                    orient="h", ax=ax, color="#91BCC9", width=.53,
                    fliersize=2.1, linewidth=1.15,
                    medianprops={"color": NAVY, "linewidth": 1.7},
                    boxprops={"edgecolor": NAVY, "alpha": .72},
                    whiskerprops={"color": MUTED}, capprops={"color": MUTED})
        small_labels = set(occupied.loc[occupied.n < 5, "short_label"])
        small = plot_data[plot_data._joint_label.isin(small_labels)]
        ycodes = {label: row for row, label in enumerate(labels)}
        for lab, grp in small.groupby("_joint_label", observed=True):
            values_small = grp._visual_value.to_numpy(dtype=float)
            ax.scatter(values_small,
                       ycodes[lab] + jitter_rng.uniform(-.13, .13, size=len(values_small)),
                       c=CORAL, s=22, alpha=.9, edgecolors="white", linewidths=.4,
                       zorder=5)
        ax.set_xlabel(f"log1p({outcome_name}) · visual only" if is_log else display_label(outcome),
                      fontsize=9.9, labelpad=8)
        ax.set_ylabel("")
        ax.set_title(outcome_name, fontsize=12, pad=16)
        ax.tick_params(axis="both", labelsize=8.5)
        ax.grid(axis="x", linewidth=.7, color="#DCE8ED")
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes_flat[n_outcomes:]:
        ax.set_visible(False)
    class_name = "Primary HRV" if subset == "primary" else "Complementary cardiac"
    figure_header(fig, pair_title(factor_a, factor_b) + f" | {class_name} distributions",
                  "All observed subgroups, no exclusions | * n<5: individual observations highlighted | Spectral log1p is display-only")
    plot_area(fig, left=0.19, right=0.98, bottom_inches=0.88, top_inches=1.52,
              wspace=0.69, hspace=0.25)
    save_figure(fig, f"{idx:02d}_{pair_slug(factor_a,factor_b)}_{subset}_distributions.png",
                pair_title(factor_a, factor_b), subset + "_distributions",
                "Source-distribution boxplots for all occupied joint cells; raw points highlighted when n<5")

for idx, (factor_a, factor_b) in enumerate(PAIR_ORDER, start=1):
    plot_distribution_atlas(factor_a, factor_b, idx, "primary")
    plot_distribution_atlas(factor_a, factor_b, idx, "complementary")
print("Total figure sheets:", len(figure_manifest))

Total figure sheets: 41


## 8. Export audit trail and autogenerated documentation

The stage README records the data contract, output schema, complete coverage, figure conventions and interpretation limits. The results report contains the cohort reference summary, the full two-factor support matrix, and the median profiles for each observed category pair. Full original-unit statistics are available in CSV, including every empty and sparsely populated cell. The project dataset and the stage 05–06 outputs are left untouched.

In [9]:
manifest = pd.DataFrame(figure_manifest)
save_table(manifest, "figure_manifest.csv")
run_information = pd.DataFrame([{
    "source_file": str(input_file), "n_records": len(data),
    "number_of_categorical_factors": len(FACTOR_LEVELS),
    "number_of_unique_pairs": len(PAIR_ORDER),
    "number_of_possible_cells": int(support_summary.n_possible_cells.sum()),
    "number_of_occupied_cells": int(support_summary.n_occupied_cells.sum()),
    "number_of_empty_cells": int(support_summary.n_empty_cells.sum()),
    "number_of_outcomes": len(MEASURES),
    "profile_rows": len(profiles), "figures": len(manifest),
    "visual_log1p_only": " | ".join(sorted(VISUAL_LOG1P)),
    "statistical_tests": "none; descriptive profiling only",
    "seed_for_small_cell_plot_jitter": SEED,
    "python": platform.python_version(), "pandas": pd.__version__,
    "numpy": np.__version__, "seaborn": sns.__version__,
    "matplotlib": matplotlib.__version__,
}])
save_table(run_information, "run_information.csv")


def formatted_markdown(frame, columns, decimals=2):
    a = frame.loc[:, columns].copy()
    for name in a.columns:
        def render(v):
            if pd.isna(v):
                return "NA"
            if isinstance(v, (int, np.integer)):
                return str(int(v))
            if isinstance(v, (float, np.floating)):
                return f"{v:.{decimals}f}"
            return str(v).replace("|", "/")
        a[name] = a[name].map(render)
    return a.to_markdown(index=False)


readme = f"""# HRV analysis 07.01 — Joint categorical profiles

## Scope
This is the first notebook of the two-factor HRV profiling phase. It produces an exhaustive **descriptive** two-factor atlas from the accepted `outputs/dataset_final.csv` and does not perform significance tests, factorial interactions, adjusted models, or conditional Spearman associations. Those questions are reserved for later stages when justified.

## Data contract
- Source: `{input_file}`
- Actual accepted records: **{len(data)}**; no rows were intentionally removed or altered.
- Original source columns: `{', '.join(FACTOR_LEVELS)}`.
- Five main HRV measures: {', '.join(PRIMARY.values())}.
- Two complementary cardiac measures: {', '.join(COMPLEMENTARY.values())}.
- Category orders and boundaries are **inherited from stage 05**; no categories are derived or merged.
- All ten distinct pairs of five factors are analyzed; see the support-summary CSV.

## Pair design
{formatted_markdown(support_summary, ['comparison','n_possible_cells','n_occupied_cells','n_empty_cells','n_occupied_cells_below_5','min_occupied_n'], 0)}

## Descriptive estimands
For every cell-outcome intersection, export actual `n`, arithmetic mean, sample SD (`ddof=1`, undefined for n<2), minimum, percentiles P05/P25/P50/P75/P95, maximum and IQR. The median is P50. Empty category intersections remain present with `n=0` and `NA` statistics. No p-values, resampling confidence intervals, effect sizes or multiple-testing corrections are generated in this notebook.

The descriptive flags `n<5` and `n<10` only disclose representation limits. They are **not exclusion thresholds** and are not claims of formal non-estimability. Do not interpret percentile boxes based on n=1 or n=2 as stable population distributions.

## Figures
- Ten cell-count heatmaps (n and within-row percentage), including empty categories.
- Ten median atlases covering all seven measures in source units. Every outcome uses its own color scale; panels are not directly color-comparable.
- Twenty distribution sheets (ten primary-HRV and ten complementary-cardiac) with all occupied subgroups, explicit n and highlighted observations for n<5.
- One overall sample-support overview.

**Total:** {len(manifest)} high-resolution multipanel figures under `figures/multipanels/`. Set `SHOW_FIGURES=True` to display them inline after saving. Spectral HF/LF/VLF may use `log1p` **only for visualization** when mathematically defined; all descriptive tables use the original source values and units.

## Generated tables
- `tables/categorical_pairwise_cell_counts.csv`: every possible categorical intersection, observed n and percentages.
- `tables/categorical_pairwise_support_summary.csv`: per-pair support, empty and small cells.
- `tables/cohort_hrv_reference.csv`: overall seven-outcome source-unit distributions.
- `tables/joint_hrv_descriptive_profiles.csv`: **complete** 7-outcome profiles for every possible two-factor category cell, including empty cells.
- `tables/figure_manifest.csv`: figure paths, scientific role and description.
- `tables/run_information.csv`: provenance and reproducibility information.

## Interpretation boundaries
This is a cross-sectional, unadjusted descriptive atlas of observed records. Group memberships are reused across factor pairs, so the resulting cell profiles are **not independent biological replications**. The cohort has imbalanced categories and empty/small intersections, which limit precision. The absence of observations does not prove biological absence. Source project structure could create dependence between records, which this notebook does not model. BMI depends on weight and height; SBP/DBP categories share physiology and do not establish clinical diagnoses. Mean RR and Mean HR are mathematically linked and not independent findings. Spectral HRV interpretation depends on the underlying measurement protocol.

## Data integrity and output behavior
No accepted input file or stage 05/06 product is overwritten. No missingness-based exclusion, threshold change, arbitrary group fusion, high-value trimming, hash assertion, or hard-coded cohort-size gate is performed. Outputs are saved only within `eda_results/hrv_associations/07_01_joint_categorical_hrv_profiles/` unless `OUTPUT_ROOT_OVERRIDE` is specified. The analytical source remains unchanged.

## Execution
1. Ensure `outputs/dataset_final.csv` exists in the project root (or set `INPUT_FILE_OVERRIDE`).
2. Run the entire notebook from top to bottom.
3. Inspect the support inventory, joint profile CSV, figure manifest, generated plots and results report.
4. Review whether targeted comparisons are scientifically justified before designing 07.02.

## Runtime
Python {platform.python_version()}, pandas {pd.__version__}, NumPy {np.__version__}, Seaborn {sns.__version__}, Matplotlib {matplotlib.__version__}. Plot jitter seed {SEED}; no statistical bootstrap applies to 07.01.
"""
(output_root / "README_07_01_joint_categorical_hrv_profiles.md").write_text(readme, encoding="utf-8")

report = [
    "# 07.01 — Joint categorical HRV profiles — Results", "",
    f"**Source records:** {len(data)} | **Categorical pairs:** {len(PAIR_ORDER)} | **Outcomes:** {len(MEASURES)} | **Figures:** {len(manifest)}", "",
    "## Interpretation boundary", "",
    "This stage is descriptive only. No tests, model interactions, or significance-based filtering are performed. Original categories and all source observations are preserved. Empty profiles do not have estimable distributions.", "",
    "## Joint category support", "",
    formatted_markdown(support_summary, ["comparison", "n_pair_complete", "n_pair_missing", "n_possible_cells", "n_occupied_cells", "n_empty_cells", "n_occupied_cells_below_5", "n_occupied_cells_below_10", "min_occupied_n"], 0), "",
    "## Overall cohort reference profiles", "",
    formatted_markdown(cohort_reference, ["outcome_label", "n", "mean", "sd", "median", "q05", "q25", "q75", "q95", "iqr"]), "",
    "## Observed cell medians and sample sizes", "",
    "All seven outcomes and all nonempty joint categories are included below. Complete means, SD, quantiles and extrema are in `tables/joint_hrv_descriptive_profiles.csv`; the table below only summarizes medians and n for readability.", "",
]
for factor_a, factor_b in PAIR_ORDER:
    view = profiles[(profiles.factor_a == factor_a) & (profiles.factor_b == factor_b) & (profiles.n > 0)].copy()
    report.extend([f"### {pair_title(factor_a, factor_b)}", "",
                   formatted_markdown(view, ["category_a", "category_b", "outcome_label", "n", "median", "q25", "q75", "support_status"]), ""])
report.extend([
    "## Figure manifest", "",
    formatted_markdown(manifest, ["figure", "role", "description"]), "",
    "## Boundaries for subsequent analyses", "",
    "The presence of an occupied cell does not establish that a within-cell statistical contrast would be precise or scientifically interpretable. Downstream 07.02 should specify comparisons before testing them; 07.03 should use continuous measures when supported. No new exclusions or category recodings are recommended by this atlas.", "",
])
(output_root / "07_01_joint_categorical_hrv_profiles_results.md").write_text("\n".join(report), encoding="utf-8")
print("Completed 07.01 at", output_root)
print("Tables:", len(list(tables_dir.glob("*.csv"))), "| Figures:", len(manifest))
print("Generated README and results report.")

Completed 07.01 at /home/david/Desktop/colabs/nunez_cristian_colabs/heart_variability_rate_profiles/eda_results/hrv_associations/07_01_joint_categorical_hrv_profiles
Tables: 6 | Figures: 41
Generated README and results report.


## 9. Interpretation and handoff

The notebook is complete once the two-factor support inventory, all category-outcome distributions and figure outputs have been inspected against the accepted dataset. This stage **does not choose inferential comparisons based on significance**; any targeted statistical questions must be justified separately in 07.02. No interaction model or new cohort definition is implied by visual differences between joint profiles.